
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Demo - Transforming PDFs to Structured Data
## Overview
In this demo, we'll use Databricks AI Functions to transform raw PDFs into structured tabular data. We'll parse documents, classify their content, and extract specific fields, turning unstructured files into queryable tables.

## Learning Objectives
By the end of this demo, you will be able to:
1. Parse a PDF using `ai_parse_document`
2. Classify parsed content using `ai_classify`
3. Extract structured fields using `ai_extract`
4. Store the results in a Delta table

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 5</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Classroom Setup

In [0]:
%run ./Includes/Classroom-Setup-04

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 8.5 MB/s eta 0:00:00
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 6.0.2
    Not uninstalling pyyaml at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-ca7065b6-01b0-4866-bde3-0e57e6f51f35
    Can't uninstall 'PyYAML'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Starting Demo Environment Setup
Using config: Includes/config/config_04.yaml

[0] Compute check passed: Serverless 5 detected.

[1] Setting up catalog...
Vocareum workspace detected.
  Catalog 'labuser17242314_1791018150' found. Using it.

[2] Setting up schema...
  Catalog : labuser17242314_1791018150
  Schema  : labuser_ka
Refreshing environment

Demo Environment Setup Complete


As a part of the workspace setup, certain assets have been managed for you (volumes, PDF, etc.). Here, we read in from the shared catalog `dbacademy`, schema `ka_all`, and volume `vol_all` the PDF `airbnb_listings.pdf`.

In [0]:
# Variables from the classroom setup
# These reference the shared volume with Airbnb PDFs
SHARED_VOLUME = "/Volumes/dbacademy/ka_all/vol_all"
LISTINGS_PDF = f"{SHARED_VOLUME}/listings/airbnb_listings.pdf"

print(f"Listings PDF:    {LISTINGS_PDF}")

Listings PDF:    /Volumes/dbacademy/ka_all/vol_all/listings/airbnb_listings.pdf


## B. Parsing Documents with `ai_parse_document`

Let's start by parsing the Airbnb listings PDF. The `ai_parse_document` function reads the binary content and returns a structured JSON representation of the document.
- We'll use the pre-defined variable from above `LISTINGS_PDF`
- We will also use `MAP('version', '2.0)`. This pins the parse output to the v2.0 schema (elements array, bbox, descriptions, etc.) instead of relying on whatever the current default version is.

1. After running the next cell, expand the **parsed** column to view the parsed PDF contents.

In [0]:
parsed_df = spark.sql(f"""
  SELECT
    path,
    ai_parse_document(content, MAP('version', '2.0')) AS parsed
  FROM READ_FILES(
    '{LISTINGS_PDF}',
    format => 'binaryFile'
  )
""")

display(parsed_df)

path parsed dbfs:/Volumes/dbacademy/ka_all/vol_all/listings/airbnb_listings.pdf {"document":{"elements":[{"bbox":[{"coord":[644,205,1056,266],"page_id":0}],"confidence":1,"content":"Airbnb Listings","description":null,"id":0,"type":"title"},{"bbox":[{"coord":[184,277,1306,312],"page_id":0}],"confidence":1,"content":"Source: sf_airbnb_listings | Column: listing_source_information | 43 records","description":null,"id":1,"type":"text"},{"bbox":[{"coord":[184,436,371,464],"page_id":0}],"confidence":1,"content":"Property ID: 958","description":null,"id":2,"type":"text"},{"bbox":[{"coord":[184,480,870,521],"page_id":0}],"confidence":1,"content":"Bright, Modern Garden Unit - 1BR/1B","description":null,"id":3,"type":"section_header"},{"bbox":[{"coord":[184,541,318,568],"page_id":0}],"confidence":1,"content":"SUMMARY","description":null,"id":4,"type":"section_header"},{"bbox":[{"coord":[184,581,1504,737],"page_id":0}],"confidence":1,"content":"New update: the house next door is under construction and there is the possibility of noise from 7am - 5pm.\nOur rates are discounted during this time period. Our bright garden unit overlooks a grassy backyard area\nwith fruit trees and native plants. It is an oasis in a big city. The apartment comfortably fits a couple or small\nfamily. It is located on a cul de sac street that ends at lovely Duboce Park.","description":null,"id":5,"type":"text"},{"bbox":[{"coord":[184,198,384,229],"page_id":1}],"confidence":1,"content":"Property ID: 7918","description":null,"id":6,"type":"text"},{"bbox":[{"coord":[184,240,1028,286],"page_id":1}],"confidence":1,"content":"A Friendly Room - UCSF/USF - San Francisco","description":null,"id":7,"type":"title"},{"bbox":[{"coord":[184,304,320,332],"page_id":1}],"confidence":1,"content":"SUMMARY","description":null,"id":8,"type":"section_header"},{"bbox":[{"coord":[184,343,1477,460],"page_id":1}],"confidence":1,"content":"Nice and good public transportation. 7 minutes walk to UCSF. 15 minutes walk to USF, ST Mary Hospital\nRoom rental-sunny view room/sink/Wi Fi (inner sunset / UCSF) (map) The place is located within walking\ndistance to Golden Gate Park, UCSF & USF","description":null,"id":9,"type":"text"},{"bbox":[{"coord":[184,198,384,229],"page_id":2}],"confidence":1,"content":"Property ID: 8142","description":null,"id":10,"type":"text"},{"bbox":[{"coord":[184,240,1170,286],"page_id":2}],"confidence":1,"content":"Friendly Room Apt. Style -UCSF/USF - San Francisco","description":null,"id":11,"type":"title"},{"bbox":[{"coord":[184,306,318,332],"page_id":2}],"confidence":1,"content":"SUMMARY","description":null,"id":12,"type":"section_header"},{"bbox":[{"coord":[184,345,1476,460],"page_id":2}],"confidence":1,"content":"Nice and good public transportation. 7 minutes walk to UCSF. 15 minutes walk to USF, ST Mary Hospital\nRoom rental-sunny view room/sink/Wi Fi (inner sunset / UCSF) (map) The place is located within walking\ndistance to Golden Gate Park, UCSF & USF","description":null,"id":13,"type":"text"},{"bbox":[{"coord":[184,198,384,229],"page_id":3}],"confidence":1,"content":"Property ID: 8339","description":null,"id":14,"type":"text"},{"bbox":[{"coord":[184,240,782,284],"page_id":3}],"confidence":1,"content":"Historic Alamo Square Victorian","description":null,"id":15,"type":"title"},{"bbox":[{"coord":[184,304,318,330],"page_id":3}],"confidence":1,"content":"SUMMARY","description":null,"id":16,"type":"section_header"},{"bbox":[{"coord":[184,343,1515,460],"page_id":3}],"confidence":1,"content":"Pls email before booking. Interior featured in magazines. Listed historic with the City of SF. Steps away from\nthe iconic Alamo Square \"Painted Ladies\". European feel, reminiscent of the Boutique Hotels in Paris,\nBerlin, Antwerp and London.","description":null,"id":17,"type":"text"},{"bbox":[{"coord":[184,198,384,229],"page_id":4}],"confidence":1,"content":"Property ID: 8739","description":null,"id":18,"type":"text"},{"bbox":[{"coord":[184,240,857,284],"page_id":4}],"conf

### B1. Exploring the Parsed Output

The parsed output is a single `VARIANT` column containing a nested JSON structure. To work with individual content blocks, we need to flatten it. The query below uses two key SQL patterns:

- **`LATERAL variant_explode(parsed:document:elements)`**: The colon (`:`) operator navigates into the VARIANT to reach the `elements` array. `variant_explode` ([AWS](https://docs.databricks.com/aws/en/sql/language-manual/functions/variant_explode) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/variant_explode) | [GCP](https://docs.databricks.com/gcp/en/sql/language-manual/functions/variant_explode)) then converts each array element into its own row, giving us one row per content block. The `LATERAL` keyword lets it reference the `parsed` column from the same `FROM` clause.
- **`element.value:type::STRING`**: Each exploded row has a `value` column (the array element as VARIANT). The `:` operator drills into fields like `type` and `content`, and `::STRING` casts the result to a string.

In [0]:
elements_df = spark.sql(f"""
  WITH parsed_docs AS (
    SELECT
      path,
      ai_parse_document(content, MAP('version', '2.0')) AS parsed
    FROM READ_FILES(
      '{LISTINGS_PDF}',
      format => 'binaryFile'
    )
  )
  SELECT
    path,
    element.value:type::STRING AS element_type,
    element.value:content::STRING AS content
  FROM
    parsed_docs,
    LATERAL variant_explode(parsed_docs.parsed:document:elements) AS element
  WHERE element.value:content IS NOT NULL
""")

display(elements_df)

path,element_type,content
dbfs:/Volumes/dbacademy/ka_all/vol_all/listings/airbnb_listings.pdf,title,Airbnb Listings
dbfs:/Volumes/dbacademy/ka_all/vol_all/listings/airbnb_listings.pdf,text,Source: sf_airbnb_listings | Column: listing_source_information | 43 records
dbfs:/Volumes/dbacademy/ka_all/vol_all/listings/airbnb_listings.pdf,text,Property ID: 958
dbfs:/Volumes/dbacademy/ka_all/vol_all/listings/airbnb_listings.pdf,section_header,"Bright, Modern Garden Unit - 1BR/1B"
dbfs:/Volumes/dbacademy/ka_all/vol_all/listings/airbnb_listings.pdf,section_header,SUMMARY
dbfs:/Volumes/dbacademy/ka_all/vol_all/listings/airbnb_listings.pdf,text,New update: the house next door is under construction and there is the possibility of noise from 7am - 5pm. Our rates are discounted during this time period. Our bright garden unit overlooks a grassy backyard area with fruit trees and native plants. It is an oasis in a big city. The apartment comfortably fits a couple or small family. It is located on a cul de sac street that ends at lovely Duboce Park.
dbfs:/Volumes/dbacademy/ka_all/vol_all/listings/airbnb_listings.pdf,text,Property ID: 7918
dbfs:/Volumes/dbacademy/ka_all/vol_all/listings/airbnb_listings.pdf,title,A Friendly Room - UCSF/USF - San Francisco
dbfs:/Volumes/dbacademy/ka_all/vol_all/listings/airbnb_listings.pdf,section_header,SUMMARY
dbfs:/Volumes/dbacademy/ka_all/vol_all/listings/airbnb_listings.pdf,text,"Nice and good public transportation. 7 minutes walk to UCSF. 15 minutes walk to USF, ST Mary Hospital Room rental-sunny view room/sink/Wi Fi (inner sunset / UCSF) (map) The place is located within walking distance to Golden Gate Park, UCSF & USF"


## C. Classifying Content with `ai_classify`

Now let's classify each parsed element. We'll categorize them into types like property descriptions, rules, pricing info, etc. A few things to note in the query below:

- **`LENGTH(...) > 50`**: Filters out short elements (headers, page numbers, etc.) that don't carry enough text for meaningful classification.
- **`ai_classify(content, '["property_description", ...]')`**: The second argument is a JSON array of candidate labels. The AI model reads the text and picks the best match.
- **`cls:response[0]`**: `ai_classify` returns a VARIANT with a `response` array. The `[0]` index retrieves the top predicted label.

In [0]:
classified_df = spark.sql(f"""
WITH elements AS (
  SELECT
    element.value:content::STRING AS content
  FROM (
    SELECT ai_parse_document(content, MAP('version', '2.0')) AS parsed
    FROM READ_FILES('{LISTINGS_PDF}', format => 'binaryFile')
  ) t,
  LATERAL variant_explode(parsed:document:elements) AS element
  WHERE element.value:content IS NOT NULL
    AND LENGTH(element.value:content::STRING) > 50
),
classified AS (
  SELECT
    content,
    ai_classify(
      content,
      '["property_description","pricing_information","location_details","amenities","host_information","other"]'
    ) AS cls
  FROM elements
)
SELECT
  content,
  cls:response[0] AS category
FROM classified
LIMIT 20
""")
display(classified_df)

content,category
Source: sf_airbnb_listings | Column: listing_source_information | 43 records,"""other"""
New update: the house next door is under construction and there is the possibility of noise from 7am - 5pm. Our rates are discounted during this time period. Our bright garden unit overlooks a grassy backyard area with fruit trees and native plants. It is an oasis in a big city. The apartment comfortably fits a couple or small family. It is located on a cul de sac street that ends at lovely Duboce Park.,"""property_description"""
"Nice and good public transportation. 7 minutes walk to UCSF. 15 minutes walk to USF, ST Mary Hospital Room rental-sunny view room/sink/Wi Fi (inner sunset / UCSF) (map) The place is located within walking distance to Golden Gate Park, UCSF & USF","""location_details"""
"Nice and good public transportation. 7 minutes walk to UCSF. 15 minutes walk to USF, ST Mary Hospital Room rental-sunny view room/sink/Wi Fi (inner sunset / UCSF) (map) The place is located within walking distance to Golden Gate Park, UCSF & USF","""location_details"""
"Pls email before booking. Interior featured in magazines. Listed historic with the City of SF. Steps away from the iconic Alamo Square ""Painted Ladies"". European feel, reminiscent of the Boutique Hotels in Paris, Berlin, Antwerp and London.","""property_description"""
"Welcome to ""The Mission,"" the sunniest neighborhood in town. Enjoy world-class dining, shopping, and nightlife on Valencia, or a Tartine picnic in Dolores Park, all just steps from our 100 year old home. We've been Hosts since Airbnb's 1st year (2009), and can suggest restaurants or insider sight-seeing ideas. We love our home, our hometown, and our neighborhood, and we look forward to sharing them with you. PLEASE click below to read more about this private room and bath in a shared home.","""host_information"""
"A Unique Guest Suite! A Spacious Art Filled Space [800 sq.ft., 12 ft.ceilings]on a SEPARATE floor from hosts quarters, own bedroom, Queen Real Bed, own bathroom, kitchenette, and office. Access to a city view garden for morning coffee or relaxing. Sleeps 2. Non- Smokers Only No Vaping Free Street Parking, Car can be left all day without moving. Convenient to all Bay Area attractions. Access to Highway 101 or 280 is less than 5 minutes. Mission and Dogpatch neighborhoods a 10-minute walk.","""property_description"""
This property is only available with a minimum stay of one month. This historic 3 bedroom Victorian house sits on a hillside overlooking the City and Marin Headlands to the north from a large private roof-top deck and it has a great garden.,"""property_description"""
"This property is only available with a minimum stay of one month. This San Francisco Victorian house is centrally located in the heart of trendy Hayes Valley within walking distance from wine bars, cafes, five star restaurants, high-end boutiques, clubs, bars, movie theaters, Civic Center, and much more.","""location_details"""
This property is only available with a minimum stay of one month. Brand new residential building incorporating touches of the City's rich architectural history and the best of modern design.,"""property_description"""


## D. Extracting Structured Fields with `ai_extract`

Let's extract specific fields from the parsed listing text: property name, location, property type, and amenities. Note:

- **`LENGTH(...) > 100`**: Uses a higher threshold than `ai_classify` because extraction needs enough surrounding text to locate meaningful field values.
- **`ai_extract(content, '["property_name", "neighborhood", ...]')`**: The second argument lists the field names you want. The AI model scans the text and returns a JSON object with those keys populated.
- **`ext:response.property_name`**: The result VARIANT has a `response` object. Use dot notation to access each extracted field by name.

In [0]:
extracted_df = spark.sql(f"""
WITH elements AS (
  SELECT
    element.value:content::STRING AS content
  FROM (
    SELECT ai_parse_document(content, MAP('version', '2.0')) AS parsed
    FROM READ_FILES('{LISTINGS_PDF}', format => 'binaryFile')
  ) t,
  LATERAL variant_explode(parsed:document:elements) AS element
  WHERE element.value:content IS NOT NULL
    AND LENGTH(element.value:content::STRING) > 100
),
extracted AS (
  SELECT
    content,
    ai_extract(
      content,
      '["property_name", "neighborhood", "property_type", "key_amenities"]'
    ) AS ext
  FROM elements
)
SELECT
  content,
  ext:response.property_name   AS property_name,
  ext:response.neighborhood    AS neighborhood,
  ext:response.property_type   AS property_type,
  ext:response.key_amenities   AS key_amenities
FROM extracted
LIMIT 15
""")

display(extracted_df)

content,property_name,neighborhood,property_type,key_amenities
New update: the house next door is under construction and there is the possibility of noise from 7am - 5pm. Our rates are discounted during this time period. Our bright garden unit overlooks a grassy backyard area with fruit trees and native plants. It is an oasis in a big city. The apartment comfortably fits a couple or small family. It is located on a cul de sac street that ends at lovely Duboce Park.,"{""value"":""bright garden unit""}","{""value"":""Duboce Park""}","{""value"":""apartment""}","{""value"":""bright garden unit, grassy backyard area, fruit trees, native plants, cul de sac street, overlooks Duboce Park""}"
"Nice and good public transportation. 7 minutes walk to UCSF. 15 minutes walk to USF, ST Mary Hospital Room rental-sunny view room/sink/Wi Fi (inner sunset / UCSF) (map) The place is located within walking distance to Golden Gate Park, UCSF & USF","{""value"":""sunny view room""}","{""value"":""Inner Sunset""}","{""value"":""Room rental""}","{""value"":""Public transportation, 7 minutes walk to UCSF, 15 minutes walk to USF and ST Mary Hospital, walking distance to Golden Gate Park, sunny view, sink, Wi-Fi""}"
"Nice and good public transportation. 7 minutes walk to UCSF. 15 minutes walk to USF, ST Mary Hospital Room rental-sunny view room/sink/Wi Fi (inner sunset / UCSF) (map) The place is located within walking distance to Golden Gate Park, UCSF & USF","{""value"":""Room rental-sunny view room/sink/Wi Fi (inner sunset / UCSF)""}","{""value"":""UCSF""}","{""value"":""Room rental""}","{""value"":""public transportation, 7 minutes walk to UCSF, 15 minutes walk to USF, ST Mary Hospital, sunny view, sink, Wi Fi, Golden Gate Park""}"
"Pls email before booking. Interior featured in magazines. Listed historic with the City of SF. Steps away from the iconic Alamo Square ""Painted Ladies"". European feel, reminiscent of the Boutique Hotels in Paris, Berlin, Antwerp and London.","{""value"":null}","{""value"":""Alamo Square""}","{""value"":null}","{""value"":""Interior featured in magazines, Listed historic with the City of SF, Steps away from Alamo Square Painted Ladies, European feel, reminiscent of Boutique Hotels in Paris, Berlin, Antwerp and London""}"
"Welcome to ""The Mission,"" the sunniest neighborhood in town. Enjoy world-class dining, shopping, and nightlife on Valencia, or a Tartine picnic in Dolores Park, all just steps from our 100 year old home. We've been Hosts since Airbnb's 1st year (2009), and can suggest restaurants or insider sight-seeing ideas. We love our home, our hometown, and our neighborhood, and we look forward to sharing them with you. PLEASE click below to read more about this private room and bath in a shared home.","{""value"":null}","{""value"":""The Mission""}","{""value"":""private room and bath in a shared home""}","{""value"":""world-class dining, shopping, and nightlife on Valencia, Tartine picnic in Dolores Park, 100 year old home, Hosts since Airbnb's 1st year (2009), can suggest restaurants or insider sight-seeing ideas""}"
"A Unique Guest Suite! A Spacious Art Filled Space [800 sq.ft., 12 ft.ceilings]on a SEPARATE floor from hosts quarters, own bedroom, Queen Real Bed, own bathroom, kitchenette, and office. Access to a city view garden for morning coffee or relaxing. Sleeps 2. Non- Smokers Only No Vaping Free Street Parking, Car can be left all day without moving. Convenient to all Bay Area attractions. Access to Highway 101 or 280 is less than 5 minutes. Mission and Dogpatch neighborhoods a 10-minute walk.","{""value"":""A Unique Guest Suite! A Spacious Art Filled Space""}","{""value"":""Mission and Dogpatch neighborhoods (10-minute walk)""}","{""value"":""Guest Suite""}","{""value"":""800 sq.ft., 12 ft. ceilings, Separate floor from hosts quarters, own bedroom, Queen Real Bed, own bathroom, kitchenette, office, city view garden access, Non-Smokers Only, No Vaping, Free Street Parking, Car can be left all day, Conv

We can also explore the raw `ai_extract` VARIANT output directly, without unpacking individual fields. This lets you see the full JSON structure returned by the function, useful for understanding the schema before deciding which fields to extract.

In [0]:
# Explore the raw ai_extract output
explore_df = spark.sql(f"""
WITH elements AS (
  SELECT
    element.value:content::STRING AS content
  FROM (
    SELECT ai_parse_document(content, MAP('version', '2.0')) AS parsed
    FROM READ_FILES('{LISTINGS_PDF}', format => 'binaryFile')
  ) t,
  LATERAL variant_explode(parsed:document:elements) AS element
  WHERE element.value:content IS NOT NULL
    AND LENGTH(element.value:content::STRING) > 100
)
SELECT
  content,
  ai_extract(
    content,
    '["property_name", "neighborhood", "property_type", "key_amenities"]'
  ) AS ext
FROM elements
LIMIT 5
""")

display(explore_df)

content,ext
New update: the house next door is under construction and there is the possibility of noise from 7am - 5pm. Our rates are discounted during this time period. Our bright garden unit overlooks a grassy backyard area with fruit trees and native plants. It is an oasis in a big city. The apartment comfortably fits a couple or small family. It is located on a cul de sac street that ends at lovely Duboce Park.,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":{""key_amenities"":{""value"":""bright garden unit, grassy backyard area with fruit trees and native plants, oasis in a big city, cul de sac street, ends at Duboce Park""},""neighborhood"":{""value"":""Duboce Park""},""property_name"":{""value"":null},""property_type"":{""value"":""garden unit""}}}"
"Nice and good public transportation. 7 minutes walk to UCSF. 15 minutes walk to USF, ST Mary Hospital Room rental-sunny view room/sink/Wi Fi (inner sunset / UCSF) (map) The place is located within walking distance to Golden Gate Park, UCSF & USF","{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":{""key_amenities"":{""value"":""sunny view, sink, Wi Fi""},""neighborhood"":{""value"":""inner sunset / UCSF""},""property_name"":{""value"":""sunny view room""},""property_type"":{""value"":""Room rental""}}}"
"Nice and good public transportation. 7 minutes walk to UCSF. 15 minutes walk to USF, ST Mary Hospital Room rental-sunny view room/sink/Wi Fi (inner sunset / UCSF) (map) The place is located within walking distance to Golden Gate Park, UCSF & USF","{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":{""key_amenities"":{""value"":""public transportation, 7 minutes walk to UCSF, 15 minutes walk to USF, ST Mary Hospital, sunny view room, sink, Wi Fi, walking distance to Golden Gate Park, UCSF & USF""},""neighborhood"":{""value"":""UCSF""},""property_name"":{""value"":""inner sunset""},""property_type"":{""value"":""Room rental""}}}"
"Pls email before booking. Interior featured in magazines. Listed historic with the City of SF. Steps away from the iconic Alamo Square ""Painted Ladies"". European feel, reminiscent of the Boutique Hotels in Paris, Berlin, Antwerp and London.","{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":{""key_amenities"":{""value"":""Interior featured in magazines, Listed historic with the City of SF, Steps away from Alamo Square Painted Ladies, European feel""},""neighborhood"":{""value"":""Alamo Square""},""property_name"":{""value"":null},""property_type"":{""value"":""Listed historic boutique hotel""}}}"
"Welcome to ""The Mission,"" the sunniest neighborhood in town. Enjoy world-class dining, shopping, and nightlife on Valencia, or a Tartine picnic in Dolores Park, all just steps from our 100 year old home. We've been Hosts since Airbnb's 1st year (2009), and can suggest restaurants or insider sight-seeing ideas. We love our home, our hometown, and our neighborhood, and we look forward to sharing them with you. PLEASE click below to read more about this private room and bath in a shared home.","{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":{""key_amenities"":{""value"":""world-class dining, shopping, and nightlife on Valencia, Tartine picnic in Dolores Park, 100 year old home, Hosts since Airbnb's 1st year (2009), private room and bath""},""neighborhood"":{""value"":""The Mission""},""property_name"":{""value"":null},""property_type"":{""value"":""private room and bath in a shared home""}}}"


## E. Storing Results in a Delta Table

Let's combine parsing, classification, and extraction into a single pipeline and save the results. The query below chains everything into one `CREATE OR REPLACE TABLE` statement:

1. The `elements` CTE parses the PDF and flattens it (same as section B1)
2. The `classified_extracted` CTE runs both `ai_classify` and `ai_extract` on each element. Since they're in the same `SELECT`, Databricks can process them in parallel per row
3. The final `SELECT` unpacks both results and writes them as named columns

Run the cell below to create a structured table from the parsed PDF content.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog_name}.{schema_name}.airbnb_structured_listings AS
WITH elements AS (
  SELECT
    element.value:content::STRING AS content
  FROM (
    SELECT ai_parse_document(content, MAP('version', '2.0')) AS parsed
    FROM READ_FILES('{LISTINGS_PDF}', format => 'binaryFile')
  ) t,
  LATERAL variant_explode(parsed:document:elements) AS element
  WHERE element.value:content IS NOT NULL
    AND LENGTH(element.value:content::STRING) > 100
),
classified_extracted AS (
  SELECT
    content,
    ai_classify(
      content,
      '["property_description","pricing_information","location_details","amenities","host_information","other"]'
    ) AS cls,
    ai_extract(
      content,
      '["property_name","neighborhood","property_type","key_amenities"]'
    ) AS ext
  FROM elements
)
SELECT
  content,
  cls:response[0]                 AS category,
  ext:response.property_name      AS property_name,
  ext:response.neighborhood       AS neighborhood,
  ext:response.property_type      AS property_type,
  ext:response.key_amenities      AS key_amenities
FROM classified_extracted
""")

print("Table created: airbnb_structured_listings")

Table created: airbnb_structured_listings


In [0]:
%sql
SELECT * FROM airbnb_structured_listings

content,category,property_name,neighborhood,property_type,key_amenities
New update: the house next door is under construction and there is the possibility of noise from 7am - 5pm. Our rates are discounted during this time period. Our bright garden unit overlooks a grassy backyard area with fruit trees and native plants. It is an oasis in a big city. The apartment comfortably fits a couple or small family. It is located on a cul de sac street that ends at lovely Duboce Park.,"""pricing_information""","{""value"":null}","{""value"":""Duboce Park""}","{""value"":""garden unit/apartment""}","{""value"":""bright garden unit, overlooks grassy backyard area with fruit trees and native plants, cul de sac street, nearby Duboce Park""}"
"Nice and good public transportation. 7 minutes walk to UCSF. 15 minutes walk to USF, ST Mary Hospital Room rental-sunny view room/sink/Wi Fi (inner sunset / UCSF) (map) The place is located within walking distance to Golden Gate Park, UCSF & USF","""location_details""","{""value"":""sunny view room""}","{""value"":""UCSF / Inner Sunset""}","{""value"":""Room rental""}","{""value"":""sunny view, sink, Wi Fi, public transportation, walking distance to UCSF, USF, ST Mary Hospital, Golden Gate Park""}"
"Nice and good public transportation. 7 minutes walk to UCSF. 15 minutes walk to USF, ST Mary Hospital Room rental-sunny view room/sink/Wi Fi (inner sunset / UCSF) (map) The place is located within walking distance to Golden Gate Park, UCSF & USF","""location_details""","{""value"":""sunny view room""}","{""value"":""inner sunset / UCSF""}","{""value"":""Room rental""}","{""value"":""sunny view, sink, Wi Fi, public transportation, walking distance to UCSF, USF, ST Mary Hospital, Golden Gate Park""}"
"Pls email before booking. Interior featured in magazines. Listed historic with the City of SF. Steps away from the iconic Alamo Square ""Painted Ladies"". European feel, reminiscent of the Boutique Hotels in Paris, Berlin, Antwerp and London.","""property_description""","{""value"":null}","{""value"":""Alamo Square""}","{""value"":""Historic Property""}","{""value"":""Magazine-featured interior, City of SF historical listing, steps from Alamo Square 'Painted Ladies', European boutique hotel ambiance""}"
"Welcome to ""The Mission,"" the sunniest neighborhood in town. Enjoy world-class dining, shopping, and nightlife on Valencia, or a Tartine picnic in Dolores Park, all just steps from our 100 year old home. We've been Hosts since Airbnb's 1st year (2009), and can suggest restaurants or insider sight-seeing ideas. We love our home, our hometown, and our neighborhood, and we look forward to sharing them with you. PLEASE click below to read more about this private room and bath in a shared home.","""location_details""","{""value"":null}","{""value"":""The Mission""}","{""value"":""Private room and bath in a shared home""}","{""value"":""world-class dining, shopping, and nightlife on Valencia; Tartine picnic in Dolores Park; 100 year old home; local hosts since Airbnb's 1st year (2009); restaurant suggestions; insider sight-seeing ideas""}"
"A Unique Guest Suite! A Spacious Art Filled Space [800 sq.ft., 12 ft.ceilings]on a SEPARATE floor from hosts quarters, own bedroom, Queen Real Bed, own bathroom, kitchenette, and office. Access to a city view garden for morning coffee or relaxing. Sleeps 2. Non- Smokers Only No Vaping Free Street Parking, Car can be left all day without moving. Convenient to all Bay Area attractions. Access to Highway 101 or 280 is less than 5 minutes. Mission and Dogpatch neighborhoods a 10-minute walk.","""property_description""","{""value"":""A Unique Guest Suite! A Spacious Art Filled Space""}","{""value"":""Mission and Dogpatch""}","{""value"":""Guest Suite""}","{""value"":""800 sq.ft., 12 ft. ceilings, Separate floor from hosts quarters, own bedroom, Queen Real Bed, own bathroom, kitchenette, office, Access to city view garden, Sleeps 2, Non- Smokers Only, No Vaping, Free Street Parking, Car

## F. Conclusion
In this demo, you:

1. **Parsed** PDFs using `ai_parse_document` to extract structured content
2. **Classified** parsed text into categories using `ai_classify`
3. **Extracted** structured fields using `ai_extract`
4. **Stored** the results in a Delta table for downstream analysis

This **Parse → Classify → Extract** pattern is ideal for turning unstructured documents into structured, queryable data. In the next demo, we'll explore the other pipeline: **Parse → Prep Search → AI Search** for building a retrieval index.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
